# Basic Prompting

To build effective prompts—especially for **Retrieval-Augmented Generation (RAG)**—it's helpful to understand the core components that make up a prompt.

Although prompt design varies depending on the application, most RAG systems are built around the following four components:

## 1. Rules (System Instructions)

This section defines how the LLM should behave throughout the conversation. It establishes the assistant's role, response style, constraints, and any task-specific instructions.

For chat-based models, these instructions are typically placed in the **system prompt**.

**Example:**
> You are an AI technical support assistant. Answer questions clearly and accurately using only the provided context. If the context does not contain enough information, state that you don't know instead of making assumptions.

---

## 2. Context (Retrieved Knowledge)

The **context** is what makes a RAG application different from a standard LLM. It contains external information retrieved from sources such as:

- Vector databases
- Company documentation
- Knowledge bases
- Search engines
- SQL databases

This retrieved information is supplied to the model so it can generate responses grounded in factual data rather than relying only on its pretrained knowledge.

In chat applications, the context is commonly inserted before the user's question.

**Example:**
> Product Documentation:
>
> - Premium members receive unlimited cloud storage.
> - Free users receive 5 GB of storage.
> - Premium subscriptions include priority customer support.

---

## 3. Question (User Input)

The **question** is the user's request or query.

This component is almost always provided by the user and is typically included as a **user message** in chat-based applications.

**Example:**
> What storage limit does a free account have?

---

## 4. Answer (Model Response)

The **answer** is the response generated by the LLM after considering:

- The system instructions
- The retrieved context
- The user's question

This is the final output returned to the user.

**Example:**
> Free accounts include **5 GB** of cloud storage. If you need unlimited storage, you'll need to upgrade to a Premium subscription.

## 1. Creating a Prompt Template

LangChain's `ChatPromptTemplate` turns a raw string into a reusable template with `{placeholders}` that get filled in at runtime (here: `context` and `query`).


In [1]:
prompt = """
Answer the user's query based on the context below.
If you cannot answer the question using the
provided information answer with "I don't know".

Context: {context}
"""


In [2]:
from langchain_core.prompts import ChatPromptTemplate

# passing the template to the LangChain model
prompt_template = ChatPromptTemplate.from_messages([
    ("system", prompt),
    ("user", "{query}"),
])

**Inspecting a template:**
- `input_variables` → lists the placeholders the template expects
- `format_messages(...)` → renders the final messages once values are supplied


In [3]:
prompt_template.input_variables

['context', 'query']

In [8]:
prompt_template.format_messages

<bound method ChatPromptTemplate.format_messages of ChatPromptTemplate(input_variables=['context', 'query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the user\'s query based on the context below.\nIf you cannot answer the question using the\nprovided information answer with "I don\'t know".\n\nContext: {context}\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})])>

## 2. Alternative: Message-Specific Templates

Instead of tuples like `("system", prompt)`, you can build each message explicitly with `SystemMessagePromptTemplate` / `HumanMessagePromptTemplate`. Same result, more explicit control over each message type.


In [4]:
from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)

prompt_template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(prompt),
    HumanMessagePromptTemplate.from_template("{query}"),
])

In [5]:
prompt_template.input_variables

['context', 'query']

In [7]:
prompt_template.format_messages

<bound method ChatPromptTemplate.format_messages of ChatPromptTemplate(input_variables=['context', 'query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the user\'s query based on the context below.\nIf you cannot answer the question using the\nprovided information answer with "I don\'t know".\n\nContext: {context}\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})])>

## 3. Setting Up the LLM

Configure the model that will actually receive the prompt — here, Groq's `llama-3.3-70b-versatile` via `ChatGroq`. `temperature=0.0` keeps responses deterministic/accurate.


In [ ]:
import os

os.environ["GROQ_API_KEY"] = "put yours"


model="llama-3.3-70b-versatile"

In [10]:
from langchain_groq import ChatGroq


# For normal accurate responses
llm = ChatGroq(temperature=0.0, model=model)

## 4. Building the Chain (LCEL)

LangChain Expression Language (`|`) chains components together: a dict mapping inputs → the `prompt_template` → the `llm`. This is the standard **prompt → model** pipeline pattern.


In [11]:
pipeline = (
    {
        "query": lambda x: x["query"],
        "context": lambda x: x["context"]
    }
    | prompt_template
    | llm
)

### Running the Pipeline — RAG Example

Supplying real `context` + a `query` demonstrates the full Rules → Context → Question → Answer flow from the intro.


In [12]:
context = """
Airline Travel Policy

Flight Changes
- Economy tickets can be changed up to 24 hours before departure.
- A change fee may apply depending on the fare type.
- Business and First Class tickets can be changed without any change fee.
- If the new flight costs more than the original ticket, the passenger must pay the fare difference.
- Tickets cannot be changed after the flight has departed.

Refunds
- Refund eligibility depends on the fare conditions selected during booking.
- Non-refundable tickets are not eligible for a cash refund.
"""

In [15]:
query = "Can I change my flight after booking?"

response = pipeline.invoke({'query':query,'context':context})

print(response.content)

Yes, you can change your flight after booking, but the conditions and fees vary depending on the type of ticket you have. Economy tickets can be changed up to 24 hours before departure, with a possible change fee, while Business and First Class tickets can be changed without a change fee.


## 5. Few-Shot Prompting

Few-shot prompting shows the model example input/output pairs before the real request, teaching it the expected pattern (here: natural language → SQL) without any fine-tuning.


In [19]:
# few shot prompt example 
fewshot_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])

In [17]:
examples = [
    {
        "input": "Find all employees older than 30.",
        "output": "SELECT * FROM employees WHERE age > 30;"
    },
    {
        "input": "Count all customers.",
        "output": "SELECT COUNT(*) FROM customers;"
    },
    {
        "input": "Show all products that cost less than 50.",
        "output": "SELECT * FROM products WHERE price < 50;"
    },
    {
        "input": "List all orders placed in 2025.",
        "output": "SELECT * FROM orders WHERE order_date >= '2025-01-01' AND order_date < '2026-01-01';"
    }
]

`FewShotChatMessagePromptTemplate` takes the example pairs and an `example_prompt` format, then renders all examples together as one block of message history.


In [20]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=fewshot_prompt,
    examples=examples,
)

# here is the formatted prompt
print(few_shot_prompt.format())

Human: Find all employees older than 30.
AI: SELECT * FROM employees WHERE age > 30;
Human: Count all customers.
AI: SELECT COUNT(*) FROM customers;
Human: Show all products that cost less than 50.
AI: SELECT * FROM products WHERE price < 50;
Human: List all orders placed in 2025.
AI: SELECT * FROM orders WHERE order_date >= '2025-01-01' AND order_date < '2026-01-01';


The examples are inserted between the system instruction and the real user request, so the model sees the pattern immediately before answering.


In [25]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "Convert the user's request into SQL."),
    few_shot_prompt,
    ("human", "{input}"),
])


In [26]:
pipeline2=(final_prompt|llm)

pipeline2.invoke({'input':"get all emp >50 and male "})

AIMessage(content="SELECT * FROM employees WHERE age > 50 AND gender = 'Male';", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 176, 'total_tokens': 192, 'completion_time': 0.027139711, 'completion_tokens_details': None, 'prompt_time': 0.008698905, 'prompt_tokens_details': None, 'queue_time': 0.051008492, 'total_time': 0.035838616}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f2c2a-9ff5-7f32-94ad-4bea46f4f544-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 176, 'output_tokens': 16, 'total_tokens': 192})

## 6. Chain-of-Thought (CoT) Prompting


### Without Chain-of-Thought

The system prompt explicitly forbids explanation, forcing a direct (often less accurate) answer to a multi-step problem.


In [27]:
no_cot_system_prompt = """
Be a helpful assistant and answer the user's question.

You MUST answer the question directly without any other
text or explanation.
"""

no_cot_prompt_template = ChatPromptTemplate.from_messages([
    ("system", no_cot_system_prompt),
    ("user", "{query}"),
])

In [28]:
query = (
    "How many keystrokes are needed to type the numbers from 1 to 500?"
)

no_cot_pipeline = no_cot_prompt_template | llm
no_cot_result = no_cot_pipeline.invoke({"query": query}).content
print(no_cot_result)

615


### With Explicit Chain-of-Thought

Chain-of-Thought (CoT) prompting instructs the model to break the problem into subproblems, solve each one, then combine them — improving accuracy on reasoning-heavy questions.


In [29]:
# Define the chain-of-thought prompt template
cot_system_prompt = """
Be a helpful assistant and answer the user's question.

To answer the question, you must:

- List systematically and in precise detail all
  subproblems that need to be solved to answer the
  question.
- Solve each sub problem INDIVIDUALLY and in sequence.
- Finally, use everything you have worked through to
  provide the final answer.
"""

cot_prompt_template = ChatPromptTemplate.from_messages([
    ("system", cot_system_prompt),
    ("user", "{query}"),
])

cot_pipeline = cot_prompt_template | llm

In [31]:
result = cot_pipeline.invoke({'query':query})

result.pretty_print()

================================== Ai Message ==================================

To solve this problem, let's break it down into subproblems.

### Subproblem 1: Identify the pattern of keystrokes for single, double, and triple-digit numbers.
- Single-digit numbers (1-9) require 1 keystroke each.
- Double-digit numbers (10-99) require 2 keystrokes each.
- Triple-digit numbers (100-500) require 3 keystrokes each.

### Subproblem 2: Calculate the number of keystrokes for single-digit numbers.
There are 9 single-digit numbers (1-9). Since each requires 1 keystroke, the total keystrokes for single-digit numbers are 9 * 1 = 9.

### Subproblem 3: Calculate the number of keystrokes for double-digit numbers.
There are 90 double-digit numbers (10-99). Since each requires 2 keystrokes, the total keystrokes for double-digit numbers are 90 * 2 = 180.

### Subproblem 4: Calculate the number of keystrokes for triple-digit numbers up to 500.
There are 401 triple-digit numbers from 100 to 500 (since 5

**Note:** Many modern LLMs apply step-by-step reasoning internally by default, even without an explicit CoT instruction in the prompt.


### Default Behavior Example

A plain system prompt with no explicit reasoning instructions — useful as a baseline to compare against the CoT and no-CoT variants above.


In [32]:
system_prompt = """
Be a helpful assistant and answer the user's question.
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "{query}"),
])

pipeline = prompt_template | llm

### Comparing Outputs

Run this alongside the earlier no-CoT / CoT examples to compare reasoning depth and answer quality.


In [33]:
result = pipeline.invoke({'query':query})

result.pretty_print()

================================== Ai Message ==================================

To determine the number of keystrokes needed to type the numbers from 1 to 500, we need to consider the number of digits in each number.

1. **Single-digit numbers (1-9)**: There are 9 single-digit numbers, and each requires 1 keystroke. So, the total keystrokes for single-digit numbers are 9.

2. **Double-digit numbers (10-99)**: There are 90 double-digit numbers, and each requires 2 keystrokes. So, the total keystrokes for double-digit numbers are 90 * 2 = 180.

3. **Triple-digit numbers (100-500)**: There are 401 triple-digit numbers, and each requires 3 keystrokes. So, the total keystrokes for triple-digit numbers are 401 * 3 = 1203.

Now, let's add up the total keystrokes:
9 (single-digit) + 180 (double-digit) + 1203 (triple-digit) = 1392

Therefore, 1392 keystrokes are needed to type the numbers from 1 to 500.
